In [0]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
import mlflow, time


spark = SparkSession.builder \
    .appName("SyntheticDataGen") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()


STORAGE_ACCOUNT = "adlssupplychain"
CONTAINER       = "supplychain-data"

STORAGE_KEY = dbutils.secrets.get(scope="adls-scope", key="adls-account-key")

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    STORAGE_KEY
)


BASE_PATH    = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
BRONZE       = f"{BASE_PATH}bronze"
SILVER_DELTA = f"{BASE_PATH}silver/supply_chain_100gb"
DLQ_TABLE    = f"{BASE_PATH}quarantine/dead_letter_queue"


spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.files.maxPartitionBytes", str(128 * 1024 * 1024))

print(f"Direct Access configured. Tuning applied for {STORAGE_ACCOUNT}.")

Direct Access configured. Tuning applied for adlssupplychain.


In [0]:


base_df = spark.read.csv(
    f"{BRONZE}/DataCoSupplyChainDataset.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    encoding="ISO-8859-1"
)

logs_df = spark.read.csv(
    f"{BRONZE}/tokenized_access_logs.csv",
    header=True,
    inferSchema=True,
    encoding="ISO-8859-1"
)

base_count = base_df.count()
logs_count = logs_df.count()
print(f"Base supply chain rows : {base_count:,}")
print(f"Access log rows        : {logs_count:,}")
print(f"Supply chain columns   : {len(base_df.columns)}")

Base supply chain rows : 180,519
Access log rows        : 469,977
Supply chain columns   : 53


In [0]:

# 180,519 base rows × 600 batches ≈ 108M rows ≈ 100 GB in Delta
REP_FACTOR = 600
print(f"Replication factor     : {REP_FACTOR}")
print(f"Estimated total rows   : {base_count * REP_FACTOR:,}")

#  3. Numeric columns to receive Gaussian noise 
# Identified from actual DataCo schema — excludes ID/flag cols
NUMERIC_NOISE_COLS = [
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Benefit per order",
    "Sales per customer",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Product Price",
    "Order Item Profit Ratio",
    "Order Item Quantity",
    "Sales",
    "Order Item Total",
    "Order Profit Per Order",
    "Product Price",
    "Latitude",
    "Longitude",
]

TIMESTAMP_COLS = [
    "order date (DateOrders)",
    "shipping date (DateOrders)",
]

Replication factor     : 600
Estimated total rows   : 108,311,400


In [0]:
#  4. Variance injection function 
def replicate_with_variance(df, batch_id: int):
    """
    Generates one synthetic batch with:
    - Gaussian noise on numeric cols (σ = 3%)
    - Cyclic timestamp shift (0–180 days, seeded by batch)
    - FK-preserving ID mutation (offset by batch × range)
    - 4% late delivery flag flip for label variance
    - 2% order status randomisation
    """
    NOISE_SIGMA = 0.03
    seed        = batch_id * 137    # prime-based seed, reproducible

    df_batch = df

    # Gaussian noise on numeric columns
    for col_name in NUMERIC_NOISE_COLS:
        if col_name in df.columns:
            df_batch = df_batch.withColumn(
                col_name,
                F.col(col_name) * (
                    1.0 + F.randn(seed=seed + abs(hash(col_name)) % 9999)
                    * NOISE_SIGMA
                )
            )
        # Profit ratio must stay in [-1, 1]
    if "Order Item Profit Ratio" in df_batch.columns:
        df_batch = df_batch.withColumn(
            "Order Item Profit Ratio",
            F.greatest(F.lit(-1.0),
                F.least(F.lit(1.0),
                    F.col("Order Item Profit Ratio")))
        )

    # Discount rate must stay in [0, 1]
    if "Order Item Discount Rate" in df_batch.columns:
        df_batch = df_batch.withColumn(
            "Order Item Discount Rate",
            F.greatest(F.lit(0.0),
                F.least(F.lit(1.0),
                    F.col("Order Item Discount Rate")))
        )

    # Shipping days must stay positive
    if "Days for shipping (real)" in df_batch.columns:
        df_batch = df_batch.withColumn(
            "Days for shipping (real)",
            F.greatest(F.lit(1.0),
                F.col("Days for shipping (real)"))
        )

    if "Days for shipment (scheduled)" in df_batch.columns:
        df_batch = df_batch.withColumn(
            "Days for shipment (scheduled)",
            F.greatest(F.lit(1.0),
                F.col("Days for shipment (scheduled)"))
        )

    # Cap real shipping at 5x scheduled max (prevents delay ratio flag)
    if ("Days for shipping (real)" in df_batch.columns and
        "Days for shipment (scheduled)" in df_batch.columns):
        df_batch = df_batch.withColumn(
            "Days for shipping (real)",
            F.least(
                F.col("Days for shipping (real)"),
                F.col("Days for shipment (scheduled)") * 4.9
            )
        )        

    # Timestamp shift: cyclic 0–180 days
    days_shift = (batch_id * 7) % 180
    for ts_col in TIMESTAMP_COLS:
        if ts_col in df.columns:
            df_batch = df_batch.withColumn(
                ts_col,
                F.date_format(
                    F.to_timestamp(
                        F.col(ts_col), "M/d/yyyy H:mm"
                    ) + F.expr(f"INTERVAL {days_shift} DAYS"),
                    "M/d/yyyy H:mm"
                )
            )

    # FK-preserving ID mutation
    # Customer Id cycles across 50K IDs per batch
    df_batch = (df_batch
        .withColumn("Customer Id",
            (F.col("Customer Id") % 50000) + (batch_id * 50000))
        .withColumn("Order Customer Id",
            (F.col("Order Customer Id") % 50000) + (batch_id * 50000))
        .withColumn("Order Id",
            F.col("Order Id") + (batch_id * 1_000_000))
        .withColumn("Order Item Id",
            F.col("Order Item Id") + (batch_id * 1_000_000))
        # Product IDs stay fixed — same catalogue, different orders
    )

    # Realistic late-delivery label flip (4% of rows)
    df_batch = df_batch.withColumn(
        "Late_delivery_risk",
        F.when(
            F.rand(seed=seed) < 0.04,
            (1 - F.col("Late_delivery_risk").cast("int")).cast("double")
        ).otherwise(F.col("Late_delivery_risk"))
    )

    # Order status variance (2% of rows)
    status_options = ["COMPLETE", "PENDING", "CLOSED",
                      "ON_HOLD", "PROCESSING", "SUSPECTED_FRAUD"]
    df_batch = df_batch.withColumn(
        "Order Status",
        F.when(
            F.rand(seed=seed + 1) < 0.02,
            F.array([F.lit(s) for s in status_options]).getItem(
                (F.rand(seed=seed + 2) * len(status_options)).cast("int"))
        ).otherwise(F.col("Order Status"))
    )

    return df_batch.withColumn("_batch_id", F.lit(batch_id))

In [0]:
# 5. Generate all batches and union 
print(f"\nGenerating {REP_FACTOR} batches...")
batches      = [replicate_with_variance(base_df, i) for i in range(REP_FACTOR)]
synthetic_df = batches[0]
for b in batches[1:]:
    synthetic_df = synthetic_df.union(b)

# 200 partitions tuned for D4s_v3 cluster (8 executors × 25)
synthetic_df = synthetic_df.repartition(200)
print("Union complete. Applying Data Quality Gate...")


Generating 600 batches...


/databricks/spark/python/pyspark/sql/column.py:464: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


Union complete. Applying Data Quality Gate...


In [0]:
# 6. DATA QUALITY GATE 
def apply_quality_gate(df):

    df = df.withColumn(
        "_dq_failure_reason",
        F.when(
            F.col("Days for shipping (real)") < 0,
            F.lit("NEGATIVE_SHIPPING_DAYS")
        ).when(
            F.col("Days for shipment (scheduled)") < 0,
            F.lit("NEGATIVE_SCHEDULED_DAYS")
        ).when(
            F.col("Sales") < 0,
            F.lit("NEGATIVE_SALES")
        ).when(
            F.col("Order Item Quantity") <= 0,
            F.lit("ZERO_OR_NEGATIVE_QUANTITY")
        ).when(
            F.col("Order Item Product Price") < 0,
            F.lit("NEGATIVE_PRICE")
        ).when(
            F.col("Order Profit Per Order").isNull(),
            F.lit("NULL_PROFIT")
        ).when(
            F.col("Order Id").isNull(),
            F.lit("NULL_ORDER_ID")
        ).when(
            F.col("Customer Id").isNull(),
            F.lit("NULL_CUSTOMER_ID")
        ).when(
            # Real shipping days should not be 5x the scheduled days
            F.col("Days for shipping (real)") >
            F.col("Days for shipment (scheduled)") * 5,
            F.lit("IMPLAUSIBLE_DELAY_RATIO")
        ).when(
            # Discount rate must be between 0 and 1
            (F.col("Order Item Discount Rate") < 0) |
            (F.col("Order Item Discount Rate") > 1),
            F.lit("INVALID_DISCOUNT_RATE")
        ).when(
            # Profit ratio must be between -1 and 1
            (F.col("Order Item Profit Ratio") < -1) |
            (F.col("Order Item Profit Ratio") > 1),
            F.lit("INVALID_PROFIT_RATIO")
        ).when(
            (F.col("Latitude") < -90) | (F.col("Latitude") > 90),
            F.lit("INVALID_LATITUDE")
        ).when(
            (F.col("Longitude") < -180) | (F.col("Longitude") > 180),
            F.lit("INVALID_LONGITUDE")
        ).otherwise(F.lit(None).cast("string"))
    )

    clean_df = (df
        .filter(F.col("_dq_failure_reason").isNull())
        .drop("_dq_failure_reason")
    )
    quarantine_df = (df
        .filter(F.col("_dq_failure_reason").isNotNull())
        .withColumn("_quarantine_ts", F.current_timestamp())
    )
    return clean_df, quarantine_df

clean_df, quarantine_df = apply_quality_gate(synthetic_df)

clean_count = clean_df.count()
bad_count   = quarantine_df.count()
total_count = clean_count + bad_count

print(f"\nData Quality Gate Results:")
print(f"  Total rows   : {total_count:,}")
print(f"  Clean        : {clean_count:,}  ({100*clean_count/total_count:.2f}%)")
print(f"  Quarantined  : {bad_count:,}  ({100*bad_count/total_count:.2f}%)")

if bad_count > 0:
    print("\nQuarantine breakdown:")
    quarantine_df \
        .groupBy("_dq_failure_reason") \
        .count() \
        .orderBy(F.desc("count")) \
        .show(truncate=False)



Data Quality Gate Results:
  Total rows   : 108,311,400
  Clean        : 108,311,398  (100.00%)
  Quarantined  : 2  (0.00%)

Quarantine breakdown:
+------------------+-----+
|_dq_failure_reason|count|
+------------------+-----+
|INVALID_LONGITUDE |2    |
+------------------+-----+



In [0]:
#  6b. Sanitise column names for Delta compatibility 
# Delta Lake rejects spaces, parentheses, and special characters.
# Replace all problematic characters with underscores.

import re

def sanitise_col_name(name: str) -> str:
    """
    Replaces spaces, parentheses, and other special characters
    with underscores. Strips leading/trailing underscores.
    """
    cleaned = re.sub(r'[ ,;{}()\n\t=]+', '_', name)
    cleaned = cleaned.strip('_')
    return cleaned

# Build rename mapping — only rename cols that actually need it
original_cols  = clean_df.columns
rename_map     = {c: sanitise_col_name(c) for c in original_cols}

# Show what will be renamed
changed = {k: v for k, v in rename_map.items() if k != v}
print(f"Renaming {len(changed)} columns:")
for old, new in changed.items():
    print(f"  '{old}'  →  '{new}'")

# Apply renames
for old_name, new_name in rename_map.items():
    if old_name != new_name:
        clean_df = clean_df.withColumnRenamed(old_name, new_name)

# Apply same renames to quarantine df
for old_name, new_name in rename_map.items():
    if old_name != new_name:
        quarantine_df = quarantine_df.withColumnRenamed(
                            old_name, new_name)

print("\nColumn names after sanitisation:")
print(clean_df.columns)

Renaming 48 columns:
  'Days for shipping (real)'  →  'Days_for_shipping_real'
  'Days for shipment (scheduled)'  →  'Days_for_shipment_scheduled'
  'Benefit per order'  →  'Benefit_per_order'
  'Sales per customer'  →  'Sales_per_customer'
  'Delivery Status'  →  'Delivery_Status'
  'Category Id'  →  'Category_Id'
  'Category Name'  →  'Category_Name'
  'Customer City'  →  'Customer_City'
  'Customer Country'  →  'Customer_Country'
  'Customer Email'  →  'Customer_Email'
  'Customer Fname'  →  'Customer_Fname'
  'Customer Id'  →  'Customer_Id'
  'Customer Lname'  →  'Customer_Lname'
  'Customer Password'  →  'Customer_Password'
  'Customer Segment'  →  'Customer_Segment'
  'Customer State'  →  'Customer_State'
  'Customer Street'  →  'Customer_Street'
  'Customer Zipcode'  →  'Customer_Zipcode'
  'Department Id'  →  'Department_Id'
  'Department Name'  →  'Department_Name'
  'Order City'  →  'Order_City'
  'Order Country'  →  'Order_Country'
  'Order Customer Id'  →  'Order_Customer_I

In [0]:
#  7. Write to Delta (Silver layer) 
mlflow.set_experiment("/supply-chain-pipeline")

with mlflow.start_run(run_name="synthetic_data_generation"):
    mlflow.log_param("replication_factor",    REP_FACTOR)
    mlflow.log_param("base_row_count",        base_count)
    mlflow.log_param("noise_sigma",           0.03)
    mlflow.log_param("timestamp_shift_days",  180)
    mlflow.log_param("dq_rules_count",        12)
    mlflow.log_param("partition_cols",        "Market, Order Region")
    mlflow.log_param("compute_type",          "Standard_D4s_v3")

    t0 = time.time()

    # Write clean data — partitioned by Market + Order Region
    (clean_df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("Market", "Order_Region")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("delta.autoOptimize.autoCompact",   "true")
        .save(SILVER_DELTA)
    )

    # Write bad rows to Dead Letter Queue
    if bad_count > 0:
        (quarantine_df.write
            .format("delta")
            .mode("append")
            .save(DLQ_TABLE)
        )

    elapsed = time.time() - t0

    mlflow.log_metric("write_time_sec",   elapsed)
    mlflow.log_metric("clean_row_count",  clean_count)
    mlflow.log_metric("quarantine_count", bad_count)
    mlflow.log_metric("pass_rate_pct",    round(100 * clean_count / total_count, 2))

    print(f"\nSilver Delta written in {elapsed:.0f}s")

#  8. Register as Hive table ─
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS supply_chain_100gb
    USING DELTA
    LOCATION '{SILVER_DELTA}'
""")
print("Registered as Hive table: supply_chain_100gb")

# Verify
spark.sql("DESCRIBE DETAIL supply_chain_100gb") \
     .select("numFiles", "sizeInBytes", "partitionColumns") \
     .show(truncate=False)

In [0]:
STORAGE_ACCOUNT = "adlssupplychain"
CONTAINER       = "supplychain-data"
BASE_ABFSS      = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

SILVER_ABFSS    = f"{BASE_ABFSS}/silver/supply_chain_100gb"
WAREHOUSE_ABFSS = f"{BASE_ABFSS}/hive-warehouse"

# Step 1: Create a database/schema with explicit ADLS location
# This tells Hive to store ALL metadata for this schema on ADLS
# instead of trying to use /user/hive/warehouse (DBFS root)
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS supply_chain_db
    LOCATION '{WAREHOUSE_ABFSS}/supply_chain_db.db'
""")
print("Database created.")

# Step 2: Register table inside that database
spark.sql("DROP TABLE IF EXISTS supply_chain_db.supply_chain_100gb")

spark.sql(f"""
    CREATE TABLE supply_chain_db.supply_chain_100gb
    USING DELTA
    LOCATION '{SILVER_ABFSS}'
""")
print("Table registered.")

# Step 3: Verify
spark.sql("DESCRIBE DETAIL supply_chain_db.supply_chain_100gb") \
     .select("numFiles", "sizeInBytes", "partitionColumns") \
     .show(truncate=False)

print(f"Row count: {spark.table('supply_chain_db.supply_chain_100gb').count():,}")

Database created.
Table registered.
+--------+-----------+----------------------+
|numFiles|sizeInBytes|partitionColumns      |
+--------+-----------+----------------------+
|59      |17103766914|[Market, Order_Region]|
+--------+-----------+----------------------+

Row count: 101,840,453
